# VI posterior vs spin-up reference — `more_sliding`

## If the Jupyter kernel dies

Matplotlib often OOM-kills the kernel on Mac. **Use the terminal script instead** (fast, reliable):

```bash
cd "/Users/anvitakallam/Ice Dynamics"
python3 scripts/validate_vi_posterior_more_sliding.py
```

Metrics only (no plots): `python3 scripts/validate_vi_posterior_more_sliding.py --no-plots`

Figures → `outputs/figures/vi/more_sliding/`

---

**Notebook:** run cells 1–3 for metrics only. Set `ENABLE_PLOTS = True` in cell 1 only if you want inline plots (may be slow / unstable).

In [ ]:
import os
import time
from pathlib import Path

import h5py
import numpy as np

ENABLE_PLOTS = False  # True may kill Jupyter kernel on Mac — use terminal script for plots

def find_root() -> Path:
    for parent in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (parent / "scripts" / "project_paths.py").exists():
            return parent
    raise FileNotFoundError("repo root not found")

ROOT = find_root()
ARCHIVE = ROOT / "Archive" if (ROOT / "Archive" / "run_torch.cfg").exists() else ROOT
NPZ_PATH = ROOT / "outputs/spinup/production/more_sliding/SteadyState_more_sliding_10500yr_ramp4000_1refine_grid.npz"
H5_PATH = ARCHIVE / "outputs/more_sliding_posterior_samples_torch.h5"
FIG_DIR = ROOT / "outputs/figures/vi/more_sliding"

print("ROOT:", ROOT)
print("HDF5:", H5_PATH.exists(), H5_PATH)
print("NPZ:", NPZ_PATH.exists(), NPZ_PATH)

### Pull HDF5 from cluster (if missing locally)

Run on your **Mac**:

```bash
rsync -avz t-9akall@login.ds:~/ice-dynamics/Archive/outputs/more_sliding_posterior_samples_torch.h5 \
           "$(pwd)/Archive/outputs/"
```

In [ ]:
_t0 = time.perf_counter()
if not H5_PATH.exists():
    raise FileNotFoundError(f"Missing predict output: {H5_PATH}")
if not NPZ_PATH.exists():
    raise FileNotFoundError(f"Missing spin-up NPZ: {NPZ_PATH}")

required = ("x", "y", "geom_mask", "eta_mean", "eta_std", "u_hat", "v_hat")
with h5py.File(H5_PATH) as f:
    missing = [k for k in required if k not in f]
    if missing:
        raise KeyError(f"HDF5 missing keys: {missing}")
    x = f["x"][...]
    y = f["y"][...]
    geom = f["geom_mask"][...].astype(bool)
    eta_mean = f["eta_mean"][...]
    eta_std = f["eta_std"][...]
    u_hat = f["u_hat"][...]
    v_hat = f["v_hat"][...]
    h5_attrs = dict(f.attrs)

with np.load(NPZ_PATH) as z:
    eta_ref = z["viscosity"]
    ux = z["ux"]
    uy = z["uy"]
    geom_ref = np.isfinite(z["surface"]) & np.isfinite(z["thickness"]) & np.isfinite(z["bed"])

eval_mask = geom & geom_ref & np.isfinite(eta_mean) & np.isfinite(eta_ref) & (eta_ref > 0) & (eta_mean > 0)
X, Y = np.meshgrid(x, y)

print("HDF5 verification: PASS")
print("attrs:", h5_attrs)
print(f"grid: {eta_mean.shape[0]} x {eta_mean.shape[1]}")
print(f"eval pixels: {eval_mask.sum():,} / {eval_mask.size:,} ({100 * eval_mask.mean():.1f}%)")
print(f"load: {time.perf_counter() - _t0:.2f}s")

In [ ]:
def log10_safe(v):
    out = np.full(v.shape, np.nan, dtype=float)
    m = v > 0
    out[m] = np.log10(v[m])
    return out

log_eta_pred = log10_safe(eta_mean[eval_mask])
log_eta_ref = log10_safe(eta_ref[eval_mask])
rel_eta = (eta_mean[eval_mask] - eta_ref[eval_mask]) / eta_ref[eval_mask]

metrics = {
    "log10_eta_rmse": float(np.sqrt(np.mean((log_eta_pred - log_eta_ref) ** 2))),
    "log10_eta_bias": float(np.mean(log_eta_pred - log_eta_ref)),
    "log10_eta_r": float(np.corrcoef(log_eta_pred, log_eta_ref)[0, 1]),
    "rel_eta_rmse": float(np.sqrt(np.mean(rel_eta ** 2))),
    "rel_eta_mae": float(np.mean(np.abs(rel_eta))),
}

speed_hat = np.hypot(u_hat, v_hat)
speed_ref = np.hypot(ux, uy)
speed_mask = eval_mask & (speed_ref > 5.0)
speed_rel = (speed_hat[speed_mask] - speed_ref[speed_mask]) / speed_ref[speed_mask]
metrics["speed_rel_rmse_gt5"] = float(np.sqrt(np.mean(speed_rel ** 2)))
metrics["speed_rel_mae_gt5"] = float(np.mean(np.abs(speed_rel)))

print("=== eta vs spin-up reference ===")
for k, v in metrics.items():
    print(f"  {k:22s} {v: .4f}")

In [ ]:
if not ENABLE_PLOTS:
    print("Skipping plots (ENABLE_PLOTS=False). Run:")
    print('  python3 scripts/validate_vi_posterior_more_sliding.py')
else:
    import gc
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import LogNorm, TwoSlopeNorm

    os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib_cache"))
    STRIDE = 2
    xs, ys = x[::STRIDE], y[::STRIDE]
    Xp, Yp = np.meshgrid(xs, ys)
    em = eval_mask[::STRIDE, ::STRIDE]

    fig, ax = plt.subplots(figsize=(6, 3))
    vals = np.concatenate([eta_ref[em], eta_mean[em]])
    norm = LogNorm(vmin=max(np.percentile(vals, 2), 1e-3), vmax=np.percentile(vals, 98))
    ax.pcolormesh(Xp / 1e3, Yp / 1e3, np.where(em, eta_mean[::STRIDE, ::STRIDE], np.nan), norm=norm, shading="auto", cmap="magma")
    ax.set_title("VI mean eta")
    plt.tight_layout()
    if FIG_DIR:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / "eta_maps_notebook.png", dpi=100)
    plt.show()
    plt.close("all")
    gc.collect()
    print("plot done")

In [ ]:
# Plots: use terminal script (Jupyter + matplotlib often kills the kernel on Mac)
print("Open PNGs after running:")
print("  python3 scripts/validate_vi_posterior_more_sliding.py")
print(f"  open {FIG_DIR}")

## Interpretation notes

- **η metrics** are the primary VI success criterion (did we recover spin-up viscosity?).
- **Velocity maps** check the PINN mean field; large errors often mean pretrain/joint data fit issues, not VI sampling.
- SSA training used **fixed basal friction C** from `cfg_json`; `lambda_mean` in the HDF5 is not used in the SSA physics loss.
- If η is systematically low/high, inspect joint training plots (`plot_vi_training.ipynb`) and `phys_scale` / η prior settings before re-running predict.